In [1]:
# ===== CELL 1 — INSTALLS (datasets PINNED: tydiqa/squad_bn are script datasets, broken on 3.x) =====
import subprocess, sys
for p in ["transformers>=4.44","sentencepiece","accelerate>=0.30","bitsandbytes","datasets==2.19.0","tqdm"]:
    subprocess.run([sys.executable,"-m","pip","install","-q",p],check=False)
print("ok")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 33.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.0/542.0 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.0/172.0 kB 11.9 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
tpot 1.1.0 requires dill>=0.3.9, but you have dill 0.3.8 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.3.1 which is incompatible.


ok


In [2]:
# ===== CELL 2 — CONFIG · SEEDS · SECRETS =====
import os, re, gc, glob, json, random, unicodedata, warnings, time
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
from dataclasses import dataclass
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_cosine_schedule_with_warmup
warnings.filterwarnings("ignore"); T0=time.time()
SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
DEVICE="cuda" if torch.cuda.is_available() else "cpu"

def get_hf_token():
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        return None
HF_TOKEN=get_hf_token()

@dataclass
class CFG:
    comp_dir:str="/kaggle/input/competitions/bengali-hallucination"
    nli_tsv:str="/kaggle/input/datasets/ajmainmahtab/bangla-natural-language-inference-dataset/NLI Dataset - Combined.tsv"
    wiki_dir:str="/kaggle/input/datasets/disisbig/bengali-wikipedia-articles"
    hf_squad:str="csebuetnlp/squad_bn"
    hf_tydi:tuple=("tydiqa","google-research-datasets/tydiqa")
    hf_ixnli:str="Divyanshu/indicxnli"
    hf_qa_70k:str="rasheduzzaman/Bangla_question_answer_pair_70K_dataset"
    
    # --- NEW BANGLA HALLU EVAL DATASETS ---
    bhe_qa_1000:str="/kaggle/input/datasets/mahdihasanqurishi/banglahallueval-qa/banglahallueval_qa_1000.csv"
    bhe_qa_full:str="/kaggle/input/datasets/mahdihasanqurishi/banglahallueval-qa/banglahallueval_qa_dataset.csv"
    bhe_qa_ds_1000:str="/kaggle/input/datasets/mahdihasanqurishi/banglahallueval-qa/banglahallueval_qa_dataset_1000.csv"
    
    local_squad:str=""; local_tydi:str=""; local_ixnli:str=""; local_qa_70k:str=""
    backbones:tuple=(("banglabert_large","csebuetnlp/banglabert_large"),)
    n_seeds_mdeberta:int=2
    llm_id:str="md-nishat-008/TigerLLM-9B-it"
    max_len:int=256; batch_size:int=16; epochs:int=4; lr:float=8e-6; warmup:float=0.15
    focal_gamma:float=1.0; focal_alpha:float=1.0
    n_wiki_files:int=8000; max_train_rows:int=110000 # Increased limit to fit new data
    use_retrieval:bool=True; n_passages:int=250000; retr_topk:int=3
    use_llm_judge:bool=True; judge_dual_prompt:bool=False
    n_boot:int=200
    
cfg=CFG()
def tleft(): print(f"[t+{(time.time()-T0)/60:.1f}m]")
print("device:",DEVICE,"| gpus:",torch.cuda.device_count(),"| HF token:",("set" if HF_TOKEN else "None (ok)"))

device: cuda | gpus: 2 | HF token: None (ok)


In [3]:
# ===== CELL 3 — BENGALI UTILS =====
BN_DIGITS="০১২৩৪৫৬৭৮৯"; BN2ASCII={ord(b):str(i) for i,b in enumerate(BN_DIGITS)}
NULLS={"[null]","null","none","nan","n/a",""}; _P=set("।,.?!;:\"'()[]{}<>/\\|-–—’‘“”…%°")
BN_CHAR=re.compile(r"[\u0980-\u09FF]"); DIGIT_RE=re.compile(r"[০-৯0-9]+")
def norm(s): return unicodedata.normalize("NFC",str(s))
def denum(s): return norm(s).translate(BN2ASCII)
def is_no_ctx(c):
    if c is None or (isinstance(c,float) and pd.isna(c)): return True
    return str(c).strip().lower() in NULLS
def toks(s): return [t for t in "".join(" " if c in _P else c for c in denum(s)).split() if t]
def numset(s): return set(re.findall(r"\d+(?:\.\d+)?",denum(s)))
def content(s,m=2): return {t for t in toks(s) if len(t)>=m}
def contain(resp,src):
    r,s=content(resp),content(src); return len(r&s)/len(r) if r else 1.0
def sent_split(t):
    parts=re.split(r"(?<=[।!?])\s+|\n+",norm(t))
    return [p.strip() for p in parts if len(p.strip())>8]
def mostly_bengali(s):
    s=str(s); b=len(BN_CHAR.findall(s)); return b>=max(1,int(0.5*len(re.findall(r"\S",s))))
def bump_digits(a):
    def b(ch):
        if ch in BN_DIGITS: return BN_DIGITS[(BN_DIGITS.index(ch)+random.randint(1,8))%10]
        if ch.isdigit(): return str((int(ch)+random.randint(1,8))%10)
        return ch
    n="".join(b(c) for c in a); return None if n==a else n

In [4]:
# ===== CELL 4 — COMPETITION DATA =====
sample=pd.DataFrame(json.load(open(os.path.join(cfg.comp_dir,"dataset samples.json"),encoding="utf-8")))
test=pd.read_csv(os.path.join(cfg.comp_dir,"test set.csv"))
sub=pd.read_csv(os.path.join(cfg.comp_dir,"sample submission.csv"))
for df in (sample,test):
    df["no_ctx"]=df["context"].map(is_no_ctx)
    df["ctx_clean"]=df.apply(lambda r:"" if r["no_ctx"] else str(r["context"]),axis=1)
    df["premise"]=df.apply(lambda r: str(r["prompt_bn"]) if r["no_ctx"]
                           else (str(r["prompt_bn"])+" "+r["ctx_clean"]).strip(),axis=1)
    df["response"]=df["response_bn"].astype(str)
assert list(sub.columns)==["id","label"] and len(sub)==len(test)==2516
print("val:",sample.shape,"| halluc:",round((sample.label==0).mean(),3),
      "| test has/no ctx:",int((~test.no_ctx).sum()),int(test.no_ctx.sum()))

val: (299, 8) | halluc: 0.455 | test has/no ctx: 1361 1155


In [5]:
# ===== CELL 5 — NLI SOURCES (TSV + IndicXNLI-bn; token optional, local-first) =====
def load_nli_tsv():
    if not os.path.exists(cfg.nli_tsv): print("tsv missing"); return pd.DataFrame()
    raw=pd.read_csv(cfg.nli_tsv,sep="\t",on_bad_lines="skip").dropna()
    cols={c.lower():c for c in raw.columns}
    def pick(*ns):
        for n in ns:
            for lc,o in cols.items():
                if n in lc: return o
    pc,hc,lc=pick("premise","sentence1","text_a"),pick("hypothesis","sentence2","text_b"),pick("label","gold","class")
    if not(pc and hc and lc): print("cols?",list(raw.columns)); return pd.DataFrame()
    m={"entailment":1,"entail":1,"neutral":0,"contradiction":0,"contradict":0,"0":1,0:1,"1":0,1:0,"2":0,2:0}
    raw["y"]=raw[lc].astype(str).str.strip().str.lower().map(lambda v:m.get(v,np.nan))
    return pd.DataFrame({"premise":raw[pc].astype(str),"response":raw[hc].astype(str),
                         "label":raw.dropna(subset=["y"])["y"].astype(int),"src":"nli"}).dropna()
def load_indicxnli():
    try:
        from datasets import load_dataset, load_from_disk
        d=load_from_disk(cfg.local_ixnli) if cfg.local_ixnli else \
          load_dataset(cfg.hf_ixnli,"bn",split="train",token=HF_TOKEN)
        if hasattr(d,"keys") and "train" in getattr(d,"column_names",{}): d=d["train"]
        df=pd.DataFrame({"premise":d["premise"],"response":d["hypothesis"],"label":d["label"]})
        df["label"]=(df["label"]==0).astype(int)      # XNLI: 0=entail,1=neutral,2=contra
        df["src"]="ixnli"
        return df.sample(min(40000,len(df)),random_state=SEED)
    except Exception as e:
        print("IndicXNLI skipped:",str(e)[:100]); return pd.DataFrame()
nli_df=pd.concat([load_nli_tsv(),load_indicxnli()],ignore_index=True)
print("NLI total:",nli_df.shape, nli_df["label"].value_counts().to_dict() if len(nli_df) else {})

cols? ['Premise', 'Entailment', 'Neutral', 'Contradiction']


Generating train split:   0%|          | 0/392702 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5010 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2490 [00:00<?, ? examples/s]

NLI total: (40000, 4) {0: 26717, 1: 13283}


In [6]:
# ===== CELL 6 — REAL BENGALI QA & BANGLA HALLU EVAL =====
def qa_rows(items):
    by_ctx={}
    for it in items: by_ctx.setdefault(it.get("context",""),[]).append(it)
    all_ans=[str(it["answer"]) for it in items if it.get("answer")]
    rows=[]
    for ctx,grp in by_ctx.items():
        ans_list = list({str(g["answer"]) for g in grp if g.get("answer")})
        for g in grp:
            q=str(g["question"]); prem=(q+" "+str(ctx)).strip()
            if g.get("answer"):
                a=str(g["answer"]); rows.append((prem,a,1,"faithful"))
                others=[x for x in ans_list if x!=a]
                if others: rows.append((prem,random.choice(others),0,"wrong_attr"))
                if DIGIT_RE.search(a):
                    c=bump_digits(a)
                    if c: rows.append((prem,c,0,"intrinsic"))
                else:
                    for _ in range(6):
                        cand=str(random.choice(all_ans))
                        if cand!=a and cand not in ctx:
                            rows.append((prem,cand,0,"extrinsic")); break
            else:
                w=[t for t in str(ctx).split() if len(t)>2]
                if w:
                    i=random.randrange(max(1,len(w)-2))
                    rows.append((prem," ".join(w[i:i+2]),0,"unanswerable"))
    return pd.DataFrame(rows,columns=["premise","response","label","mode"])

def load_squad_bn():
    try:
        from datasets import load_dataset
        d=load_dataset(cfg.hf_squad,split="train",token=HF_TOKEN)
        return [{"context":r["context"],"question":r["question"],"answer":r["answers"]["text"][0] if r["answers"]["text"] else None} for r in d]
    except: return []
    
def load_qa_70k():
    try:
        from datasets import load_dataset
        d=load_dataset(cfg.hf_qa_70k,split="train",token=HF_TOKEN)
        return [{"context": "", "question": str(r.get("question", "")).strip(), "answer": str(r.get("answer", "")).strip()} for r in d if r.get("question") and r.get("answer")]
    except: return []

# --- DYNAMIC LOADER FOR BHE DATASETS ---
def load_bhe_datasets():
    paths = [cfg.bhe_qa_1000, cfg.bhe_qa_full, cfg.bhe_qa_ds_1000]
    rows = []
    for p in paths:
        if not os.path.exists(p): continue
        try: df = pd.read_csv(p)
        except: continue
        
        cols = {c.lower().strip(): c for c in df.columns}
        
        ctx_col = cols.get("context", cols.get("document", cols.get("ctx", None)))
        q_col = cols.get("question", cols.get("prompt", cols.get("q", cols.get("prompt_bn", None))))
        ans_col = cols.get("answer", cols.get("response", cols.get("a", cols.get("response_bn", cols.get("correct_answer", None)))))
        hallu_col = cols.get("hallucinated_answer", cols.get("hallucination", cols.get("wrong_answer", None)))
        label_col = cols.get("label", cols.get("target", cols.get("is_hallucinated", None)))
        
        for _, r in df.iterrows():
            ctx = str(r[ctx_col]) if ctx_col and pd.notna(r[ctx_col]) else ""
            if str(ctx).lower() in {"nan", "null", "[null]", "none", ""}: ctx = ""
            q = str(r[q_col]) if q_col and pd.notna(r[q_col]) else ""
            
            prem = (q + " " + ctx).strip() if ctx else q
            if not prem: continue
            
            # Case 1: Dataset has both Correct and Hallucinated columns
            if ans_col and hallu_col:
                ans, hal = str(r[ans_col]), str(r[hallu_col])
                if ans and ans.lower() != 'nan': rows.append((prem, ans, 1, "bhe_faithful"))
                if hal and hal.lower() != 'nan': rows.append((prem, hal, 0, "bhe_hallucinated"))
                    
            # Case 2: Dataset has a predefined Label column
            elif label_col and ans_col:
                ans = str(r[ans_col])
                lbl_val = str(r[label_col]).strip().lower()
                lbl = 1 if lbl_val in ['1', 'true', 'correct', 'entailment'] else (0 if lbl_val in ['0', 'false', 'hallucinated', 'contradiction', 'wrong'] else None)
                if lbl is not None and ans and ans.lower() != 'nan':
                    rows.append((prem, ans, lbl, "bhe_labeled"))
                    
            # Case 3: Pure QA pairs (No labels, assume correct)
            elif ans_col and not label_col and not hallu_col:
                ans = str(r[ans_col])
                if ans and ans.lower() != 'nan':
                    rows.append((prem, ans, 1, "bhe_positive"))
                    
    df_out = pd.DataFrame(rows, columns=["premise", "response", "label", "mode"])
    if len(df_out): df_out["src"] = "bhe_qa"
    return df_out.drop_duplicates()

qa_items=load_squad_bn()+load_qa_70k()
random.shuffle(qa_items); qa_items=qa_items[:40000]
qa_df=qa_rows(qa_items) if qa_items else pd.DataFrame(columns=["premise","response","label","mode"])
if len(qa_df): qa_df["src"]="qa"

bhe_df = load_bhe_datasets()
if len(qa_df) and len(bhe_df):
    qa_df = pd.concat([qa_df, bhe_df], ignore_index=True)
elif len(bhe_df):
    qa_df = bhe_df

print("QA + BHE total:",qa_df.shape,"| modes:",qa_df["mode"].value_counts().to_dict() if len(qa_df) else {})

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating train split:   0%|          | 0/81072 [00:00<?, ? examples/s]

QA + BHE total: (82021, 5) | modes: {'faithful': 23174, 'extrinsic': 17465, 'unanswerable': 16826, 'wrong_attr': 15351, 'intrinsic': 5709, 'bhe_positive': 3496}


In [7]:
# ===== CELL 7 — CLOZE SYNTHETIC FROM WIKI =====
STOP=set("এবং ও কিন্তু বা যে যা এই সেই তার তাদের করা হয় হয়ে হন ছিল ছিলেন একটি একটা এক থেকে সালে জন্য এর কে না নয় করে করেন দিয়ে পরে আগে মধ্যে সাথে হিসেবে".split())
def spans_in(s):
    out=[m.group() for m in DIGIT_RE.finditer(s)]; w=toks(s)
    for n in (1,2,3):
        for i in range(len(w)-n+1):
            g=w[i:i+n]
            if g[0] in STOP or g[-1] in STOP: continue
            if all(len(x)<3 for x in g): continue
            if any(DIGIT_RE.fullmatch(x) for x in g): continue
            sp=" ".join(g)
            if mostly_bengali(sp): out.append(sp)
    return list(dict.fromkeys(out))
def load_wiki(limit):
    files=[]
    for s in ("train/train","valid/valid",""): files+=glob.glob(os.path.join(cfg.wiki_dir,s,"*.txt"))
    random.shuffle(files); out=[]
    for fp in files[:limit]:
        try:
            t=open(fp,encoding="utf-8",errors="ignore").read()
            if len(t)>200 and mostly_bengali(t[:400]): out.append(t[:3000])
        except: pass
    return out
def build_cloze(passages):
    pp=[]
    for p in passages:
        sp=[]
        for s in sent_split(p): sp+=[(s,x) for x in spans_in(s)]
        pp.append(sp)
    rows=[]
    for pi,p in enumerate(passages):
        cand=pp[pi]
        if len(cand)<4: continue
        numc=[(s,x) for s,x in cand if DIGIT_RE.fullmatch(x)]
        sent,span=random.choice(numc) if (numc and random.random()<0.5) else random.choice(cand)
        if span not in sent: continue
        prem=sent.replace(span,"____",1)+" — শূন্যস্থানে কী বসবে? "+p
        rows.append((prem,span,1,"faithful"))
        if DIGIT_RE.fullmatch(span):
            c=bump_digits(span)
            if c: rows.append((prem,c,0,"intrinsic"))
        others=[x for s2,x in cand if x!=span and x not in sent]
        if others: rows.append((prem,random.choice(others),0,"wrong_attr"))
        oj=random.randrange(len(passages))
        if oj!=pi and pp[oj]: rows.append((prem,random.choice(pp[oj])[1],0,"extrinsic"))
    df=pd.DataFrame(rows,columns=["premise","response","label","mode"]); df["src"]="synth"; return df
wiki_passages=load_wiki(cfg.n_wiki_files); print("wiki passages:",len(wiki_passages))
synth_df=build_cloze(wiki_passages) if wiki_passages else pd.DataFrame(columns=["premise","response","label","mode","src"])
print("cloze synthetic:",synth_df.shape)

wiki passages: 7065
cloze synthetic: (23483, 5)


In [8]:
# ===== CELL 8 — ASSEMBLE + MODE-STRATIFIED 50/50 BALANCE =====
if len(nli_df): nli_df=nli_df.assign(mode=nli_df["src"])
parts=[d for d in (qa_df,synth_df,nli_df) if d is not None and len(d)]
train_all=pd.concat([p[["premise","response","label","mode","src"]] for p in parts],ignore_index=True).dropna()
train_all=train_all[train_all["response"].str.len()>0].drop_duplicates(subset=["premise","response"])
train_all=train_all.sample(frac=1,random_state=SEED).reset_index(drop=True)

def cap(df):
    # Prioritize keeping all real QA and BHE datasets
    keep=[df[df.src.isin(["qa", "bhe_qa"])]]
    room=cfg.max_train_rows-len(keep[0])
    for s in ("synth","nli","ixnli"):
        part=df[df.src==s]
        keep.append(part.sample(min(len(part),max(0,room)),random_state=SEED)); room-=len(keep[-1])
    return pd.concat(keep).sample(frac=1,random_state=SEED).reset_index(drop=True)
    
train_all=cap(train_all)
c1=train_all[train_all.label==1]; c0=train_all[train_all.label==0]
k=min(len(c1),len(c0))
if len(c0)>k:
    c0=(c0.groupby("mode",group_keys=False)
          .apply(lambda g:g.sample(max(1,int(round(k*len(g)/len(train_all[train_all.label==0])))),random_state=SEED)))
    c0=c0.sample(min(len(c0),k),random_state=SEED)
if len(c1)>k: c1=c1.sample(k,random_state=SEED)
train_all=pd.concat([c1,c0]).sample(frac=1,random_state=SEED).reset_index(drop=True)

n_hold=min(3000,len(train_all)//10)
synth_hold=train_all.iloc[:n_hold].reset_index(drop=True)
train_main=train_all.iloc[n_hold:].reset_index(drop=True)
print("train:",train_main.shape,"| labels:",train_main.label.value_counts().to_dict())

train: (66902, 5) | labels: {1: 33471, 0: 33431}


In [9]:
# ===== CELL 9 — DATASET · FOCAL · TRAIN/PREDICT (fp32 params + fp16 autocast) =====
class PairDS(Dataset):
    def __init__(self,df,tok,mx,lab=True):
        self.p=df["premise"].astype(str).tolist(); self.h=df["response"].astype(str).tolist()
        self.y=df["label"].tolist() if lab else None; self.t=tok; self.m=mx
    def __len__(self): return len(self.p)
    def __getitem__(self,i):
        e=self.t(self.p[i],self.h[i],truncation=True,max_length=self.m,padding="max_length",return_tensors="pt")
        it={"input_ids":e["input_ids"].squeeze(0),"attention_mask":e["attention_mask"].squeeze(0)}
        if self.y is not None: it["labels"]=torch.tensor(self.y[i],dtype=torch.long)
        return it
class Focal(nn.Module):
    def __init__(self,gamma,alpha):
        super().__init__(); self.g=gamma; self.register_buffer("w",torch.tensor([alpha,1.0]))
    def forward(self,lg,y):
        ce=F.cross_entropy(lg,y,weight=self.w.to(lg.device),reduction="none")
        pt=torch.exp(-ce); return ((1-pt)**self.g*ce).mean()
def resolve_model(name,hf):
    for c in (f"/kaggle/input/{name}",f"/kaggle/input/{name}/{name}"):
        if os.path.exists(os.path.join(c,"config.json")): return c
    return hf
@torch.no_grad()
def predict_proba(model,tok,df,mx,bs):
    model.eval(); out=[]
    for b in DataLoader(PairDS(df,tok,mx,lab=False),batch_size=bs,shuffle=False):
        with torch.amp.autocast("cuda",dtype=torch.float16):
            lg=model(input_ids=b["input_ids"].to(DEVICE),attention_mask=b["attention_mask"].to(DEVICE)).logits.float()
        out.append(torch.softmax(lg,-1)[:,1].cpu().numpy())
    return np.concatenate(out)
def train_backbone(name,hf,tr,val,seed=SEED,quiet=False):
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
    path=resolve_model(name,hf); tok=AutoTokenizer.from_pretrained(path)
    model=AutoModelForSequenceClassification.from_pretrained(
        path,num_labels=2,ignore_mismatched_sizes=True).float().to(DEVICE)
    crit=Focal(cfg.focal_gamma,cfg.focal_alpha)
    opt=torch.optim.AdamW(model.parameters(),lr=cfg.lr,weight_decay=0.01)
    ld=DataLoader(PairDS(tr.sample(frac=1,random_state=seed),tok,cfg.max_len),
                  batch_size=cfg.batch_size,shuffle=True,pin_memory=True)
    tot=len(ld)*cfg.epochs; sch=get_cosine_schedule_with_warmup(opt,int(tot*cfg.warmup),tot)
    scaler=torch.amp.GradScaler("cuda")
    for ep in range(cfg.epochs):
        model.train()
        for b in ld:
            opt.zero_grad()
            with torch.amp.autocast("cuda",dtype=torch.float16):
                lg=model(input_ids=b["input_ids"].to(DEVICE),attention_mask=b["attention_mask"].to(DEVICE)).logits
                loss=crit(lg,b["labels"].to(DEVICE))
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sch.step()
        f=f1_score(val["label"],(predict_proba(model,tok,val,cfg.max_len,cfg.batch_size*2)>=0.5).astype(int),pos_label=0)
        if not quiet: print(f"  {name} s{seed} ep{ep+1}: valF1(c0)={f:.4f}")
    return model,tok

In [10]:
# ===== CELL 10 — TRAIN: BanglaBERT-Large ONLY =====
sig_val={}; sig_test={}; keep_for_retr=None

# ডাইনামিক্যালি আপনার CFG থেকে নাম নিবে (banglabert বা banglabert_large)
bb_key = list(dict(cfg.backbones).keys())[0]

m, tk = train_backbone(bb_key, dict(cfg.backbones)[bb_key], train_main, sample)

sig_val["bb"] = predict_proba(m, tk, sample, cfg.max_len, cfg.batch_size*2)
sig_test["bb"] = predict_proba(m, tk, test, cfg.max_len, cfg.batch_size*2)

torch.save(m.state_dict(), f"/kaggle/working/{bb_key}.pt")

# mDeBERTa যেহেতু নেই, তাই BanglaBERT কেই রিট্রিভালের জন্য মেমরিতে রাখা হলো
if cfg.use_retrieval:
    keep_for_retr = (m.half().eval(), tk)
else:
    del m; gc.collect(); torch.cuda.empty_cache()

tleft()

config.json:   0%|          | 0.00/880 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/119 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.35G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/1.35G [00:00<?, ?B/s]

ElectraForSequenceClassification LOAD REPORT from: csebuetnlp/banglabert_large
Key                                               | Status     | 
--------------------------------------------------+------------+-
electra.embeddings.position_ids                   | UNEXPECTED | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
discriminator_predictions.dense.bias              | UNEXPECTED | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
discriminator_predictions.dense.weight            | UNEXPECTED | 
classifier.dense.bias                             | MISSING    | 
classifier.out_proj.bias                          | MISSING    | 
classifier.out_proj.weight                        | MISSING    | 
classifier.dense.weight                           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the ch

  banglabert_large s42 ep1: valF1(c0)=0.4574
  banglabert_large s42 ep2: valF1(c0)=0.3509
  banglabert_large s42 ep3: valF1(c0)=0.4444
  banglabert_large s42 ep4: valF1(c0)=0.4229
[t+184.6m]


In [11]:
# ===== CELL 11 — RETRIEVAL-AUGMENTED no_context (`retr`) =====
def build_retr_signal():
    if not (cfg.use_retrieval and wiki_passages and keep_for_retr):
        return np.full(len(sample),np.nan), np.full(len(test),np.nan)
    chunks=[]
    for p in wiki_passages:
        for i in range(0,len(p),400):
            c=p[i:i+500]
            if len(c)>120: chunks.append(c)
        if len(chunks)>=cfg.n_passages: break
    print("retrieval corpus:",len(chunks),"passages")
    vec=TfidfVectorizer(analyzer="char_wb",ngram_range=(3,5),max_features=200000,sublinear_tf=True)
    Mx=vec.fit_transform(chunks)
    model,tok=keep_for_retr
    def score(df):
        out=np.full(len(df),np.nan)
        idx=np.where(df["no_ctx"].values)[0]
        if len(idx)==0: return out
        Q=vec.transform(df.iloc[idx]["prompt_bn"].astype(str).tolist())
        Sim=linear_kernel(Q,Mx)
        top=np.argsort(-Sim,axis=1)[:,:cfg.retr_topk]
        prem=[]; resp=[]
        for r_,ti in zip(df.iloc[idx].itertuples(),top):
            for j in ti:
                prem.append(str(r_.prompt_bn)+" "+chunks[j]); resp.append(str(r_.response_bn))
        pp=predict_proba(model,tok,pd.DataFrame({"premise":prem,"response":resp}),cfg.max_len,cfg.batch_size*2)
        out[idx]=pp.reshape(len(idx),cfg.retr_topk).max(1); return out
    rv=score(sample); rt=score(test)
    return rv,rt
retr_val,retr_test=build_retr_signal()
if keep_for_retr: del keep_for_retr; gc.collect(); torch.cuda.empty_cache()

retrieval corpus: 26982 passages


In [12]:
# ===== CELL 12 — LEX/NUM (has_context) =====
def lexnum(df):
    p=np.array([contain(r["response_bn"],r["ctx_clean"]) for _,r in df.iterrows()])
    nu=np.array([len(numset(r["response_bn"])-numset(r["ctx_clean"])) for _,r in df.iterrows()])
    s=0.7*p+0.3*(nu==0); s[df["no_ctx"].values]=np.nan; return s
lex_val=lexnum(sample); lex_test=lexnum(test)

In [13]:
# ===== CELL 13 — TIGERLLM-9B JUDGE (DYNAMIC CATEGORY PROMPTING) =====
def run_llm_judge(df):
    if not cfg.use_llm_judge: return np.full(len(df),np.nan)
    from transformers import AutoModelForCausalLM, BitsAndBytesConfig
    import re
    
    path = resolve_model("TigerLLM-9B-it", cfg.llm_id)
    tk = AutoTokenizer.from_pretrained(path)
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                             bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
    llm = AutoModelForCausalLM.from_pretrained(path, quantization_config=bnb, device_map="auto").eval()
    dev = next(llm.parameters()).device
    
    def digit_ids(d):
        ids=set()
        for s in (d, " "+d):
            e = tk.encode(s, add_special_tokens=False)
            if e: ids.add(e[-1])
        return list(ids)
        
    ids1, ids0 = digit_ids("1"), digit_ids("0")
    
    # ১. ডেটা ক্যাটাগরি নির্ধারণকারী ফাংশন (আপডেটেড)
    def get_category(prompt_text, ctx_text, response_text):
        p_lower = str(prompt_text).lower()
        combined_text = f"{prompt_text} {ctx_text} {response_text}"
        
        # যদি লেখায় কোনো ইংরেজি অক্ষর/রোমান হরফ (a-z, A-Z) থাকে, তবে এটি Code-Mixed বা Banglish
        if re.search(r'[a-zA-Z]', combined_text):
            return "code_mixed"
        elif ctx_text and not is_no_ctx(ctx_text):
            return "comprehension"
        elif any(k in p_lower for k in ["অর্থ", "ভাবার্থ", "সমার্থক", "বিপরীত", "মানে কী"]):
            return "vocabulary"
        elif is_math_or_logic(prompt_text, ""):
            return "math"
        else:
            return "general_knowledge"

    # ২. ক্যাটাগরি অনুযায়ী সুনির্দিষ্ট এবং কঠোর System Prompt
    def build_sys_prompt(category):
        base_rule = "কোনো ব্যাখ্যা দেবেন না। শুধুমাত্র একটি সংখ্যা আউটপুট দিন।"
        
        if category == "code_mixed":
            return (f"আপনি একজন বহুভাষিক (Multilingual) এবং কোড-মিক্সড (Code-Mixed/Banglish) ডেটা বিশ্লেষক। "
                    f"এখানে প্রশ্ন, অনুচ্ছেদ বা উত্তরে বাংলা, ইংরেজি বা বাংলিশ ভাষার মিশ্রণ থাকতে পারে। "
                    f"ভাষার মিশ্রণ থাকা সত্ত্বেও মূল অর্থটি যাচাই করুন। প্রস্তাবিত উত্তরটি যদি সঠিক এবং প্রাসঙ্গিক হয় (Correct), তবে শুধুমাত্র '1' লিখুন। "
                    f"উত্তরটি যদি ভুল, বানোয়াট বা অপ্রাসঙ্গিক হয় (Hallucinated), তবে শুধুমাত্র '0' লিখুন। {base_rule}")

        elif category == "comprehension":
            return (f"আপনি একটি নির্ভুল হ্যালুসিনেশন সনাক্তকরণ এআই। আপনাকে একটি অনুচ্ছেদ, একটি প্রশ্ন এবং একটি উত্তর দেওয়া হবে। "
                    f"প্রস্তাবিত উত্তরটি যদি অনুচ্ছেদের তথ্যের সাথে সম্পূর্ণ মিলে যায় এবং বাইরের কোনো বানোয়াট তথ্য না থাকে (Correct), তবে শুধুমাত্র '1' লিখুন। "
                    f"উত্তরটি যদি অনুচ্ছেদের তথ্যের সাথে সাংঘর্ষিক হয় বা অনুচ্ছেদে নেই এমন কিছু দাবি করে (Hallucinated), তবে শুধুমাত্র '0' লিখুন। {base_rule}")
                    
        elif category == "vocabulary":
            return (f"আপনি একজন বিশেষজ্ঞ বাংলা ভাষাবিদ। আপনার কাজ হলো প্রশ্নের প্রদত্ত শব্দ বা বাগধারার অর্থ যাচাই করা। "
                    f"প্রস্তাবিত উত্তরটি যদি শব্দ বা বাগধারার সঠিক অর্থ বা ভাবার্থ প্রকাশ করে (Correct), তবে শুধুমাত্র '1' লিখুন। "
                    f"উত্তরটি যদি ভুল বা মনগড়া হয় (Hallucinated), তবে শুধুমাত্র '0' লিখুন। {base_rule}")
                    
        elif category == "math":
            return (f"আপনি একজন গাণিতিক মূল্যায়নকারী এআই। প্রস্তাবিত গাণিতিক উত্তরটি নিখুঁত হিসাব-নিকাশ মেনে চলে কি না তা যাচাই করুন। "
                    f"হিসাব ও চূড়ান্ত উত্তর সম্পূর্ণ সঠিক (Correct) হলে শুধুমাত্র '1' লিখুন। "
                    f"হিসাবে সামান্যতম ভুল বা গরমিল (Wrong) থাকলে শুধুমাত্র '0' লিখুন। {base_rule}")
                    
        else: # general_knowledge
            return (f"আপনি একজন কঠোর তথ্য-যাচাইকারী। আপনার কাজ হলো সাধারণ জ্ঞান এবং ঐতিহাসিক তথ্যের সত্যতা যাচাই করা। "
                    f"প্রস্তাবিত উত্তরটি যদি বাস্তবিকভাবে সম্পূর্ণ সত্য এবং নির্ভুল হয় (Correct), তবে শুধুমাত্র '1' লিখুন। "
                    f"উত্তরটি যদি ভুল, মিথ্যা বা বানোয়াট হয় (Hallucinated), তবে শুধুমাত্র '0' লিখুন। {base_rule}")

    def one_pass():
        out = np.zeros(len(df))
        for i, r in enumerate(df.itertuples()):
            ctx = getattr(r, "ctx_clean", "")
            
            # Category নির্ধারণের জন্য response_bn ও পাঠানো হচ্ছে
            cat = get_category(r.prompt_bn, ctx, r.response_bn)
            SYS = build_sys_prompt(cat)
            
            u = (f"CONTEXT: {ctx}\n" if ctx else "") + f"QUESTION: {r.prompt_bn}\nANSWER: {r.response_bn}\nVerdict:"
            
            # TigerLLM Chat Template
            enc = tk.apply_chat_template([{"role":"system", "content":SYS}, {"role":"user", "content":u}],
                                         add_generation_prompt=True, return_tensors="pt", return_dict=True)
            ii = enc["input_ids"][:, -768:].to(dev)
            am = enc["attention_mask"][:, -768:].to(dev)
            
            with torch.no_grad():
                lg = llm(input_ids=ii, attention_mask=am).logits[0, -1, :].float()
            
            # Extract probability of outputting '1' vs '0'
            p1 = torch.logsumexp(lg[ids1], 0)
            p0 = torch.logsumexp(lg[ids0], 0)
            out[i] = torch.softmax(torch.stack([p0, p1]), 0)[1].item()
            
            if i % 250 == 0: 
                print(f"  TigerLLM judge {i}/{len(df)} | Cat: {cat}")
        return out
        
    res = one_pass()
    del llm; gc.collect(); torch.cuda.empty_cache(); 
    return res

llm_val = run_llm_judge(sample)
llm_test = run_llm_judge(test)
tleft()

config.json:   0%|          | 0.00/916 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/1065 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 114.00 MiB. GPU 1 has a total capacity of 14.56 GiB of which 42.81 MiB is free. Including non-PyTorch memory, this process has 14.52 GiB memory in use. Of the allocated memory 14.28 GiB is allocated by PyTorch, and 121.72 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
# ===== CELL 14 — RANK-NORMALIZE SIGNALS PER REGIME (val∪test) =====
def stackX(df,sv,lex,retr,llm):
    # এখান থেকে "md":sv["md"] কলামটি বাদ দেওয়া হয়েছে
    X=pd.DataFrame({"bb":sv["bb"],"lex":lex,"retr":retr,"llm":llm})
    X["no_ctx"]=df["no_ctx"].values; return X

Xv=stackX(sample,sig_val,lex_val,retr_val,llm_val)
Xt=stackX(test,sig_test,lex_test,retr_test,llm_test)

def rank_norm(Xv,Xt):
    Xv,Xt=Xv.copy(),Xt.copy()
    for reg in (False,True):
        mv=Xv["no_ctx"].values==reg; mt=Xt["no_ctx"].values==reg
        for c in [c for c in Xv.columns if c!="no_ctx"]:
            allv=np.concatenate([Xv.loc[mv,c].values,Xt.loc[mt,c].values]).astype(float)
            ok=~np.isnan(allv)
            if ok.sum()<10: continue
            ref=np.sort(allv[ok])
            def r(x):
                y=x.astype(float); m=~np.isnan(y)
                y[m]=np.searchsorted(ref,y[m],side="right")/len(ref); return y
            Xv.loc[mv,c]=r(Xv.loc[mv,c].values); Xt.loc[mt,c]=r(Xt.loc[mt,c].values)
    return Xv,Xt

Xv,Xt=rank_norm(Xv,Xt)
yv=sample["label"].values; print("signals:",[c for c in Xv.columns if c!='no_ctx'])

In [ ]:
# ===== CELL 15 — POWELL over DETERMINISTIC BOOTSTRAP OBJECTIVE (validated) =====
from scipy.optimize import minimize
def f1c0(yy,p,t): return f1_score(yy,(p>=t).astype(int),pos_label=0)
def blend(X,w):
    cols=[c for c in X.columns if c!="no_ctx"]; P=np.zeros(len(X)); ws=np.zeros(len(X))
    for c in cols:
        v=X[c].values.astype(float); m=~np.isnan(v); P[m]+=w[c]*v[m]; ws[m]+=w[c]
    return np.where(ws>0,P/np.maximum(ws,1e-9),0.5)
def tune(Xv,yv,mask,n_boot=60,seed=SEED):
    cols=[c for c in Xv.columns if c!="no_ctx"]; Xr=Xv[mask]; yr=yv[mask]; m=len(yr)
    rng=np.random.RandomState(seed)
    B=rng.randint(0,m,size=(n_boot,m))
    Y0=(yr[B]==0).astype(np.float32)                       
    act0=Y0.sum(1,keepdims=True)
    ths=np.linspace(0.1,0.9,30).astype(np.float32)
    def boot_best_f1(p):
        P=p[B].astype(np.float32)
        L=(P[:,:,None]<ths[None,None,:]).astype(np.float32)
        tp=np.einsum("bm,bmt->bt",Y0,L)
        f1=2*tp/np.maximum(L.sum(1)+act0,1e-9)             
        return float(f1.max(1).mean())
    def objective(wa): return -boot_best_f1(blend(Xr,{c:max(w,0.0) for c,w in zip(cols,wa)}))
    res=minimize(objective,np.ones(len(cols)),method="Powell",bounds=[(0,5)]*len(cols),
                 options={"maxiter":30,"xtol":1e-2,"ftol":1e-3})
    w={c:float(v) for c,v in zip(cols,res.x)}
    p=blend(Xr,w); grid=np.quantile(p,np.linspace(0.05,0.95,60))
    rng2=np.random.RandomState(seed+1); picks=[]
    for _ in range(cfg.n_boot):
        b=rng2.randint(0,m,m); pb,yb=p[b],yr[b]
        pred0=(pb[:,None]<grid[None,:]).astype(np.float32)
        tp=((yb==0)[:,None]*pred0).sum(0)
        f1=2*tp/np.maximum(pred0.sum(0)+(yb==0).sum(),1e-9)
        picks.append(grid[int(f1.argmax())])
    t=float(np.median(picks))
    return w,t,f1c0(yr,p,t),-res.fun
wc,tc,fc,bc=tune(Xv,yv,~sample["no_ctx"].values); wn,tn,fn,bn=tune(Xv,yv,sample["no_ctx"].values)
print("has_ctx",{k:round(v,1) for k,v in wc.items()},"thr",round(tc,3),"pointF1",round(fc,4),"bootF1",round(bc,4))
print("no_ctx ",{k:round(v,1) for k,v in wn.items()},"thr",round(tn,3),"pointF1",round(fn,4),"bootF1",round(bn,4))
pv=np.where(sample["no_ctx"].values,blend(Xv,wn),blend(Xv,wc)); tv=np.where(sample["no_ctx"].values,tn,tc)
print("OVERALL valF1(c0):",round(f1_score(yv,(pv>=tv).astype(int),pos_label=0),4),
      "| all-0 floor:",round(f1_score(yv,np.zeros(len(yv)),pos_label=0),4))

In [ ]:
# ===== CELL 16 — SUBMISSION =====
pt=np.where(test["no_ctx"].values,blend(Xt,wn),blend(Xt,wc)); tt=np.where(test["no_ctx"].values,tn,tc)
out=pd.DataFrame({"id":test["id"].values,"label":(pt>=tt).astype(int)})
assert list(out.columns)==["id","label"] and len(out)==2516
assert out["label"].isin([0,1]).all() and (out["id"].values==test["id"].values).all()
out.to_csv("submission.csv",index=False)
print("submission.csv",out.shape,"| halluc rate:",round((out.label==0).mean(),3)); tleft()

In [ ]:
# ===== CELL 17 — DIAGNOSTICS =====
for reg,mask in (("has_ctx",~sample["no_ctx"].values),("no_ctx",sample["no_ctx"].values)):
    pr=(pv[mask]>=tv[mask]).astype(int)
    print(f"{reg}: n={mask.sum()} valF1(c0)={f1_score(yv[mask],pr,pos_label=0):.4f} "
          f"pred-halluc={np.mean(pr==0):.2f} true-halluc={np.mean(yv[mask]==0):.2f}")
diag=pd.DataFrame({"regime":np.where(sample.no_ctx,"no","has"),"label":yv,"p":pv})
for c in [c for c in Xv.columns if c!="no_ctx"]: diag[c]=Xv[c].values
diag.to_csv("/kaggle/working/val_signals.csv",index=False)
print("saved val_signals.csv")